# KGAS gNFW Kinematic Fitting

Fit a generalized NFW (gNFW) dark matter velocity profile to KGAS
visibilities using **UVfit** + **KinMS**.  The inner density slope
$\gamma$ is a free MCMC parameter: $\gamma = 0$ is a flat core,
$\gamma = 1$ is a classical NFW cusp.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tempfile
import os

from uvfit import UVDataset, Fitter
from uvfit.forward_model import gNFWKinMSModel, gnfw_circular_velocity
from astropy.io import fits as pyfits
from pymakeplots import pymakeplots as pmp

try:
    import corner
    HAS_CORNER = True
except ImportError:
    HAS_CORNER = False
    print("Install 'corner' for corner plots: pip install corner")

## 2. Configuration

In [ ]:
DATA_PATH = "/Users/thbrown/kilogas/DR1/visibilities/KILOGAS007.small.npz"

VSYS = 13583.0        # systemic velocity, km/s
VMAX = 200.0          # maximum rotation velocity, km/s
R_SCALE = 3.0         # scale radius, arcsec
PA_INIT = 147.4       # position angle, degrees
CELLSIZE = 0.1        # arcsec per pixel

F_REST = 230.538e9    # CO(2-1) rest frequency, Hz
C_KMS = 299792.458    # speed of light, km/s

NX = NY = 256         # spatial pixels
VEL_BUFFER = 100.0    # km/s buffer beyond +/-Vmax for trimming

## 3. Load and inspect data

In [ ]:
d = np.load(DATA_PATH)
u_all, v_all = d["u"], d["v"]
vis_all, weights_all = d["vis"], d["weights"]
freqs_all = d["freqs"]

print(f"Baselines : {u_all.shape[0]}")
print(f"Channels  : {freqs_all.shape[0]}")
print(f"Freq range: {freqs_all.min()/1e9:.4f} \u2013 {freqs_all.max()/1e9:.4f} GHz")

In [ ]:
vel_all = C_KMS * (1.0 - freqs_all / F_REST)

avg_amp = np.abs(vis_all).mean(axis=0)

fig, ax1 = plt.subplots(figsize=(10, 3))
ax1.plot(vel_all, avg_amp, "k-", lw=0.5)
ax1.axvline(VSYS, color="C1", ls="--", label=f"vsys = {VSYS:.0f} km/s")
ax1.axvline(VSYS - VMAX, color="C3", ls=":", alpha=0.6)
ax1.axvline(VSYS + VMAX, color="C3", ls=":", alpha=0.6,
            label=f"\u00b1{VMAX:.0f} km/s")
ax1.set_xlabel("Radio velocity (km/s)")
ax1.set_ylabel("Mean |V|")
ax1.legend()
plt.tight_layout()
plt.show()

## 4. Trim to velocity window

In [ ]:
v_lo = VSYS - VMAX - VEL_BUFFER
v_hi = VSYS + VMAX + VEL_BUFFER

chan_mask = (vel_all >= v_lo) & (vel_all <= v_hi)
freqs_trim = freqs_all[chan_mask]
vis_trim = vis_all[:, chan_mask]
weights_trim = weights_all[:, chan_mask]
vel_trim = vel_all[chan_mask]

dv_kms = float(np.median(np.abs(np.diff(vel_trim))))
n_chan_trim = int(chan_mask.sum())
print(f"Channel width: {dv_kms:.3f} km/s")
print(f"Velocity range: {vel_trim.min():.1f} \u2013 {vel_trim.max():.1f} km/s")
print(f"Trimmed cube will have {n_chan_trim} channels")

uvdata = UVDataset(
    u=u_all, v=v_all,
    vis_data=vis_trim, weights=weights_trim, freqs=freqs_trim,
)

## 4b. Pre-fit diagnostics

Quick checks on visibility SNR, radial uv-coverage, and physical
resolution before committing to a full MCMC run.

In [ ]:
from astropy.cosmology import Planck18
import astropy.units as au

# --- A. Integrated visibility SNR ---
amp_line = np.abs(vis_trim)
w_safe = np.where(weights_trim > 0, weights_trim, np.inf)
sigma_vis = 1.0 / np.sqrt(w_safe)
snr_per_vis = amp_line / sigma_vis
integrated_snr = float(np.sqrt(np.sum(snr_per_vis ** 2)))
snr_per_channel = np.sqrt(np.sum(snr_per_vis ** 2, axis=0))

print(f"Integrated line SNR: {integrated_snr:.1f}")
if integrated_snr < 10:
    print("⚠ WARNING: Integrated SNR < 10 — detection may be marginal")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(vel_trim, snr_per_channel, "k-", lw=0.8)
ax.axhline(10, color="C3", ls="--", alpha=0.6, label=r"SNR = 10")
ax.axvline(VSYS, color="C1", ls="--", alpha=0.5, label=f"vsys = {VSYS:.0f}")
ax.set_xlabel("Radio velocity (km/s)")
ax.set_ylabel("SNR per channel")
ax.set_title("Visibility SNR per channel")
ax.legend()
plt.tight_layout()
plt.show()

# --- B. Radial visibility profile (q-plot) ---
q_all = np.sqrt(u_all ** 2 + v_all ** 2)
q_max = float(np.max(q_all))

N_QBINS = 50
q_edges = np.linspace(float(np.min(q_all)), q_max, N_QBINS + 1)
q_centers = 0.5 * (q_edges[:-1] + q_edges[1:])
amp_binned = np.zeros(N_QBINS)
noise_floor = np.zeros(N_QBINS)

for i in range(N_QBINS):
    bl_mask = (q_all >= q_edges[i]) & (q_all < q_edges[i + 1])
    n_bl = int(bl_mask.sum())
    if n_bl == 0:
        continue
    amp_binned[i] = float(np.mean(np.abs(vis_trim[bl_mask, :]).mean(axis=1)))
    w_bin = weights_trim[bl_mask, :]
    w_bin_safe = np.where(w_bin > 0, w_bin, np.inf)
    noise_floor[i] = float(np.sqrt(np.sum(1.0 / w_bin_safe)) / n_bl)

# --- C. Critical q and high-q SNR ---
theta_core_rad = R_SCALE * np.pi / (180.0 * 3600.0)
q_crit = 1.0 / theta_core_rad

high_q_mask = (q_centers >= 0.5 * q_crit) & (q_centers <= q_max)
if high_q_mask.any() and np.any(noise_floor[high_q_mask] > 0):
    A_high = float(np.mean(amp_binned[high_q_mask]))
    sigma_high = float(np.mean(noise_floor[high_q_mask]))
    high_q_snr = A_high / sigma_high if sigma_high > 0 else np.inf
else:
    high_q_snr = 0.0

print(f"q_crit (1/θ_core):   {q_crit:.0f} wavelengths")
print(f"q_max  (long. base): {q_max:.0f} wavelengths")
print(f"High-q SNR:          {high_q_snr:.1f}")
if q_max < q_crit:
    print("⚠ WARNING: q_max < q_crit — baselines do not reach the scale radius")
if high_q_snr < 5:
    print("⚠ WARNING: High-q SNR < 5 — γ formally unconstrained at this resolution")

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(q_centers, amp_binned, "k.-", label="Mean |V|")
ax.semilogy(q_centers, 3 * noise_floor, "C3--", lw=1, label=r"$3\sigma$ noise floor")
ax.axvline(q_crit, color="C0", ls=":", lw=1.5, label=f"$q_{{\\rm crit}}$ = {q_crit:.0f}")
if high_q_mask.any():
    ax.axvspan(0.5 * q_crit, q_max, alpha=0.08, color="C0", label="High-q window")
ax.set_xlabel(r"$q = \sqrt{u^2 + v^2}$  (wavelengths)")
ax.set_ylabel("Amplitude")
ax.set_title("Radial visibility profile")
ax.legend()
plt.tight_layout()
plt.show()

# --- D. Physical resolution ---
z = VSYS / C_KMS
D_Mpc = float(Planck18.luminosity_distance(z).to(au.Mpc).value)
theta_res_rad = 1.0 / (2.0 * q_max) if q_max > 0 else np.inf
theta_res_arcsec = float(np.degrees(theta_res_rad) * 3600.0)
kpc_per_arcsec = float(Planck18.kpc_proper_per_arcmin(z).to(au.kpc / au.arcsec).value)
R_phys_kpc = theta_res_arcsec * kpc_per_arcsec
R_scale_kpc = R_SCALE * kpc_per_arcsec

print(f"\n{'─' * 45}")
print(f" Physical Resolution Summary")
print(f"{'─' * 45}")
print(f" Redshift (Hubble flow):  z = {z:.5f}")
print(f" Luminosity distance:     {D_Mpc:.1f} Mpc")
print(f" Angular resolution:      {theta_res_arcsec:.3f} arcsec")
print(f" Physical resolution:     {R_phys_kpc:.2f} kpc")
print(f" Scale radius:            {R_SCALE:.1f} arcsec = {R_scale_kpc:.2f} kpc")
print(f" kpc/arcsec:              {kpc_per_arcsec:.3f}")
print(f"{'─' * 45}")
if R_phys_kpc > R_scale_kpc:
    print(f"⚠ WARNING: Physical resolution ({R_phys_kpc:.2f} kpc) > scale radius "
          f"({R_scale_kpc:.2f} kpc) — data cannot resolve the inner profile")

## 5. gNFW velocity profiles

The generalized NFW density profile has inner slope $\gamma$ as a
continuous parameter:
$$\rho(r) \propto \frac{1}{(r/r_s)^\gamma\,(1 + r/r_s)^{3-\gamma}}$$

$\gamma = 0$ gives a flat core, $\gamma = 1$ gives classical NFW.

In [ ]:
radius = np.arange(0.01, 100, 0.1)
sbprof = np.exp(-radius / R_SCALE)

gammas = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5]
fig, ax = plt.subplots(figsize=(8, 4))
for g in gammas:
    vc = gnfw_circular_velocity(radius, VMAX, R_SCALE, g)
    ax.plot(radius, vc, label=f"$\\gamma={g}$")
ax.set_xlabel("Radius (arcsec)")
ax.set_ylabel("V$_c$ (km/s)")
ax.set_xlim(0, 30)
ax.legend()
ax.set_title(f"gNFW velocity profiles (Vmax={VMAX}, r_s={R_SCALE})")
plt.tight_layout()
plt.show()

## 6. Set up gNFW model and fit

In [ ]:
model = gNFWKinMSModel(
    vmax=VMAX, r_scale=R_SCALE, radius=radius,
    xs=NX, ys=NY, vs=n_chan_trim,
    cell_size_arcsec=CELLSIZE, channel_width_kms=dv_kms,
    sbprof=sbprof, sbrad=radius,
)
fitter = Fitter(uvdata=uvdata, forward_model=model)

init_params = {
    "inc": 15.0,
    "pa": PA_INIT,
    "flux": 1.0,
    "vsys": 0.0,
    "gas_sigma": 10.0,
    "gamma": 0.5,
}
print(f"Free parameters: {list(init_params.keys())}")
print(f"Bounds: {model.bounds}")

In [ ]:
%%time
result_grad = fitter.fit(initial_params=init_params, method="L-BFGS-B")
print(f"L-BFGS-B  rchi2={result_grad.reduced_chi2:.6f}")
print(f"  params: {result_grad.params}")

In [ ]:
%%time
N_WALKERS = 32
N_STEPS = 200
N_BURN = 50

result_mcmc = fitter.fit(
    initial_params=result_grad.params,
    method="emcee",
    n_walkers=N_WALKERS,
    n_steps=N_STEPS,
    n_burn=N_BURN,
)
print(f"emcee MAP  rchi2={result_mcmc.reduced_chi2:.6f}")
print(f"  params: {result_mcmc.params}")

## 7. Corner plot

In [ ]:
if HAS_CORNER:
    param_names = list(result_mcmc.params.keys())
    samples = result_mcmc.chains.reshape(-1, len(param_names))
    labels = ["inc", "PA", "flux", r"$v_{\rm sys}$", r"$\sigma_{\rm gas}$", r"$\gamma$"]

    fig = corner.corner(samples, labels=labels, show_titles=True, title_fmt=".3f")
    fig.suptitle("gNFW posterior", y=1.02, fontsize=14)
    plt.show()

## 8. Gamma posterior

In [ ]:
gamma_idx = param_names.index("gamma")
gamma_samples = samples[:, gamma_idx]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(gamma_samples, bins=50, density=True, color="C0", alpha=0.7)
ax.axvline(0.0, color="C2", ls="--", lw=2, label=r"$\gamma=0$ (core)")
ax.axvline(1.0, color="C3", ls="--", lw=2, label=r"$\gamma=1$ (NFW cusp)")
med = np.median(gamma_samples)
lo, hi = np.percentile(gamma_samples, [16, 84])
ax.axvline(med, color="k", ls="-", lw=1.5, label=f"median = {med:.2f}")
ax.axvspan(lo, hi, alpha=0.15, color="k", label=f"68% CI = [{lo:.2f}, {hi:.2f}]")
ax.set_xlabel(r"$\gamma$ (inner density slope)")
ax.set_ylabel("Posterior density")
ax.legend()
ax.set_title("Inner slope posterior")
plt.tight_layout()
plt.show()

## 9. Best-fit model cube (pymakeplots)

In [ ]:
def model_cube_to_fits(cube_vyx, vel_kms, cellsize_arcsec, filepath):
    """Write a (v,y,x) numpy model cube to a minimal FITS for pymakeplots."""
    nv, ny, nx = cube_vyx.shape
    dv = float(np.median(np.diff(vel_kms)))
    hdr = pyfits.Header()
    hdr['CTYPE1'] = 'RA---SIN'
    hdr['CTYPE2'] = 'DEC--SIN'
    hdr['CTYPE3'] = 'VRAD'
    hdr['CDELT1'] = -cellsize_arcsec / 3600.0
    hdr['CDELT2'] = cellsize_arcsec / 3600.0
    hdr['CDELT3'] = dv * 1000.0
    hdr['CRPIX1'] = nx / 2.0 + 0.5
    hdr['CRPIX2'] = ny / 2.0 + 0.5
    hdr['CRPIX3'] = 1.0
    hdr['CRVAL1'] = 180.0
    hdr['CRVAL2'] = 45.0
    hdr['CRVAL3'] = float(vel_kms[0]) * 1000.0
    hdr['CUNIT1'] = 'deg'
    hdr['CUNIT2'] = 'deg'
    hdr['CUNIT3'] = 'm/s'
    hdr['BMAJ'] = cellsize_arcsec / 3600.0
    hdr['BMIN'] = cellsize_arcsec / 3600.0
    hdr['BPA'] = 0.0
    hdr['BUNIT'] = 'Jy/beam'
    hdr['RESTFRQ'] = 230538000000.0
    pyfits.writeto(filepath, cube_vyx.astype(np.float32), hdr, overwrite=True)


cube_vyx = model.generate_cube(result_mcmc.params)

tmpfile = tempfile.NamedTemporaryFile(suffix='.fits', delete=False)
model_cube_to_fits(cube_vyx, vel_trim, CELLSIZE, tmpfile.name)
tmpfile.close()
try:
    p = pmp(cube=tmpfile.name)
    p.vsys = VSYS
    p.posang = result_mcmc.params["pa"]
    gamma_map = result_mcmc.params["gamma"]
    p.galname = f"gNFW ($\\gamma$={gamma_map:.2f})"
    p.make_all()
finally:
    os.unlink(tmpfile.name)